# 12 - Advanced Python


## 1. `is` vs `==`

Python'da `==` operatörü iki nesnenin **değer eşitliğini** (`__eq__` dunder metodunu çağırarak), `is` operatörü ise iki değişkenin bellekteki **aynı nesne kimliğine** (`id(a) == id(b)`) işaret edip etmediğini denetler.

**Syntax**
```python
obj_a == obj_b
obj_a is obj_b
obj_a is not obj_b
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Değer karşılaştırmaları için her zaman `==`; `None`, `True`, `False` gibi tekil (singleton) nesneleri kontrol etmek için ise `is` / `is not` kullanılır.
* **Dikkat Edilmesi Gerekenler:** CPython küçük tam sayıları (`-5` ile `256` arası) ve bazı kısa dizgileri bellekte önbelleğe alır (interning). Bu nedenle `a = 10; b = 10` için `a is b` sonucu `True` dönerken, çalışma zamanında üretilen büyük sayılarda `False` döner. Sayısal veya metinsel eşitlik için asla `is` kullanılmamalıdır.


In [ ]:
class Track:
    def __init__(self, title: str, bpm: int) -> None:
        self.title = title
        self.bpm = bpm

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Track):
            return NotImplemented
        return self.__dict__ == other.__dict__


track_a: Track = Track('Soléa', 120)
track_b: Track = Track('Soléa', 120)
track_ref: Track = track_a

print(track_a == track_b)
print(track_a is track_b)
print(track_a is track_ref)
print(id(track_a), id(track_b))

base: int = 500
x: int = base + 500
y: int = 1000
print(x == y, x is y)


## 2. Lambda Functions

`lambda` anahtar sözcüğü ile tanımlanan, isim verilmeden satır içinde oluşturulabilen ve tek bir ifadenin (expression) sonucunu otomatik olarak döndüren anonim fonksiyonlardır.

**Syntax**

```python
lambda param1, param2, /: expression
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** `sorted()`, `map()`, `filter()`, `min()`, `max()` gibi yüksek mertebeli fonksiyonlara (higher-order functions) tek seferlik basit bir dönüşüm veya sıralama anahtarı geçirmek için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Lambda gövdesinde `return`, döngü veya atama deyimleri (statements) yer alamaz; yalnızca tek bir ifade yazılabilir. PEP 8 kuralları gereği bir lambda ifadesini değişkene atamak (`fn = lambda x: ...`) yerine standart `def` bloğu kullanmak okunabilirlik açısından daha doğrudur.


In [ ]:
sample_rates: list[int] = [22050, 44100, 48000, 96000, 192000]
hd_rates: list[int] = list(filter(lambda rate: rate >= 48000, sample_rates))

plugins: list[tuple[str, float]] = [('Reverb', 4.2), ('EQ', 0.8), ('Compressor', 1.5)]
by_cpu_load: list[tuple[str, float]] = sorted(plugins, key=lambda item: item[1])

print(hd_rates)
print(by_cpu_load)


## 3. Walrus Operator (`:=`)

Python 3.8 ile tanıtılan atama ifadesi (assignment expression) operatörüdür; bir ifadeyi değerlendirirken sonucunu aynı anda bir değişkene atamayı sağlar.

**Syntax**

```python
(variable := expression)
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** `while` döngü koşullarında girdi okurken, `if` bloklarında hesaplanan bir değeri gövdede tekrar kullanırken veya list/dict comprehension içinde pahalı bir fonksiyon çağrısını iki kez çalıştırmaktan kaçınmak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Operatör önceliği nedeniyle `if` veya `while` koşullarında karşılaştırma operatörleriyle birlikte kullanıldığında parantez içine alınmalıdır: `if (n := len(data)) > 10:`. Okunabilirliği bozan iç içe kullanımlardan kaçınılmalıdır.


In [ ]:
def analyze_text(text: str) -> dict[str, int | float | list[str]]:
    return {
        'char_count': (chars := len(text.replace(' ', ''))),
        'words': (words := text.split()),
        'word_count': (count := len(words)),
        'avg_word_len': round(chars / count, 2) if count else 0.0,
    }


stats = analyze_text('Modern Python syntax improves clarity')
for k, v in stats.items():
    print(k, v, sep=': ')

raw_chunks: list[str] = ['packet_1', 'packet_2', '', 'packet_3']
iterator = iter(raw_chunks)
while chunk := next(iterator, ''):
    print(f'Processing {chunk}')


## 4. Dataclasses (`@dataclass`)

Veri tutmaya odaklı sınıflar için `__init__`, `__repr__` ve `__eq__` gibi basmakalıp (boilerplate) dunder metotlarını tip ipuçlarına bakarak otomatik üreten sınıf dekoratörüdür.

**Syntax**
```python
from dataclasses import dataclass, field

@dataclass(frozen=False, slots=False, kw_only=False)
class ModelName:
    attr: type = default_value
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Konfigürasyon nesneleri, DTO'lar (Data Transfer Objects) ve yapılandırılmış veri modelleri oluştururken kullanılır.
* **Dikkat Edilmesi Gerekenler:** Değiştirilebilir (mutable) varsayılan değerler (`list`, `dict`) doğrudan sınıf gövdesinde `items: list[str] = []` şeklinde tanımlanamaz (`ValueError` verir); bunun yerine `field(default_factory=list)` kullanılmalıdır. Python 3.10+ ile gelen `slots=True` ve `kw_only=True` parametreleri bellek tüketimini azaltır ve API güvenliğini artırır.


In [ ]:
from dataclasses import dataclass, field


@dataclass(slots=True)
class Preset:
    name: str
    gain_db: float = 0.0
    tags: list[str] = field(default_factory=list)


p1: Preset = Preset('Warm Tube', -1.5, ['analog', 'vocal'])
p2: Preset = Preset('Warm Tube', -1.5, ['analog', 'vocal'])

print(p1)
print(p1 == p2)


## 5. Generators (`yield`)

Tüm veri kümesini belleğe tek seferde yüklemek yerine, `yield` anahtar sözcüğü ile her çağrıda sıradaki tek bir değeri üreten ve yürütme durumunu dondurup koruyan tembel (lazy) yineleyicilerdir.

**Syntax**
```python
def gen_func(n: int):
    yield value

gen_expr = (expr for item in iterable)
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Gigabaytlarca büyüklükteki log dosyalarını satır satır okurken, sonsuz sayı dizileri üretirken veya veri işleme hatları (pipelines) kurarken bellek tüketimini $O(1)$ düzeyinde tutmak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Üreteçler tek kullanımlıktır (exhaustible); tüm elemanlar tüketildikten sonra tekrar `next()` çağrılırsa `StopIteration` istisnası fırlar. Üzerinde `list(gen)` çağırmak tüm veriyi belleğe yükleyeceği için bellek avantajını ortadan kaldırır.


In [ ]:
from collections.abc import Generator


def frame_stream(total_frames: int) -> Generator[int, None, None]:
    for frame in range(total_frames):
        yield frame


stream = frame_stream(10**12)
first_batch: list[int] = [next(stream) for _ in range(5)]
next_frame: int = next(stream)

squared_gen = (n * n for n in range(1, 6))

print(first_batch)
print(next_frame)
print(list(squared_gen))


**Daha yeni bir kullanım olarak:**

```python
def frame_stream(total_frames: int) -> Generator[int, None, None]:
    yield from range(total_frames)

```


## 6. `del` Keyword

Mevcut isim alanından (namespace) bir değişken bağını kaldırmak veya değiştirilebilir (mutable) koleksiyonlardan indeks/dilim/anahtar bazlı eleman silmek için kullanılan deyimdir.

**Syntax**
```python
del variable_name
del sequence[index]
del mapping[key]
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Büyük veri yapılarıyla iş bittiğinde referansı serbest bırakarak çöp toplayıcının (garbage collector) belleği geri kazanmasını sağlamak veya sözlük/liste içinden belirli öğeleri kaldırmak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** `del` doğrudan nesneyi bellekten yok etmez; yalnızca o ismin nesneye olan referansını siler. Nesnenin referans sayacı (reference count) sıfıra düştüğünde bellek serbest bırakılır. Silinen bir değişkene tekrar erişmeye çalışmak `NameError` fırlatır.


In [ ]:
nodes: list[str] = ['alpha', 'beta', 'gamma', 'delta']
del nodes[1]
print(nodes)

config: dict[str, int] = {'retries': 3, 'timeout': 30, 'debug_port': 9229}
del config['debug_port']
print(config)

temp_buffer: list[int] = [1, 2, 3]
del temp_buffer


## 7. `assert` Statement

Geliştirme ve hata ayıklama (debug) aşamasında programın iç mantığına dair değişmez varsayımları (invariants) doğrulayan, koşul `False` olduğunda `AssertionError` fırlatan hata ayıklama aracıdır.

**Syntax**
```python
assert condition, 'Optional error message'
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Birim testlerinde (unit tests) ve geliştirme sırasında asla gerçekleşmemesi gereken dahili yazılım hatalarını erken yakalamak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Python yorumlayıcısı optimize modda (`python -O` veya `python -OO`) çalıştırıldığında `__debug__` sabiti `False` olur ve tüm `assert` ifadeleri bayt kodundan tamamen çıkarılır. Bu nedenle kullanıcı girdi doğrulaması, yetkilendirme kontrolleri veya yan etkili (side-effect) işlemler asla `assert` ile yapılmamalıdır; bunların yerine `if ...: raise ValueError(...)` kullanılmalıdır.


In [ ]:
def load_manifest(payload: dict[str, str]) -> str:
    assert isinstance(payload, dict), 'Payload must be a dictionary'
    assert len(payload) > 0, 'Manifest cannot be empty'
    return f'Loaded {len(payload)} entries'


print(__debug__)
print(load_manifest({'version': '1.0.0'}))


## 8. `match-case` (Structural Pattern Matching)

Python 3.10 ile eklenen yapısal örüntü eşleştirme mekanizmasıdır; bir değerin yalnızca eşitliğini değil, veri yapısının biçimini (liste, demet, sözlük, sınıf öznitelikleri) ayrıştırarak koşul kontrolü yapmayı sağlar.

**Syntax**
```python
match subject:
    case pattern_1:
        ...
    case pattern_2 | pattern_3 if guard_condition:
        ...
    case _:
        ...
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Komut ayrıştırıcıları (CLI parsers), JSON/API yanıt yapıları, durum makineleri (state machines) ve çok dallı karmaşık veri ayrıştırma senaryolarında kullanılır.
* **Dikkat Edilmesi Gerekenler:** `_` joker karakteri (wildcard) hiçbir değişkene atama yapmadan her şeyi yakalar ve son dal olarak yazılmalıdır. `|` operatörü ile alternatif örüntüler birleştirilebilir; `if` koruyucusu (guard) ise örüntü eşleştikten sonra ek mantıksal koşul denetler.


In [ ]:
def parse_command(raw: str) -> str:
    tokens: list[str] = raw.split()
    match tokens:
        case ['render', target]:
            return f'Rendering single target: {target}'
        case ['delete' | 'remove', *files] if len(files) >= 2:
            return f'Batch deleting {len(files)} files: {files}'
        case ['delete' | 'remove', _]:
            return 'Error: Batch delete requires at least 2 files'
        case _:
            return 'Unknown command'


print(parse_command('render intro.mp4'))
print(parse_command('delete a.wav b.wav c.wav'))
print(parse_command('remove a.wav'))


## 9. Decorators

Mevcut bir fonksiyonun veya sınıfın kaynak kodunu değiştirmeden ona öncesinde veya sonrasında çalışan ek davranışlar kazandıran, bir callable alıp yeni bir callable döndüren üst düzey fonksiyonlardır.

**Syntax**
```python
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Loglama, çalışma süresi ölçümü, yetkilendirme kontrolleri, önbellekleme ve yeniden deneme (retry) gibi kesişen ilgileri (cross-cutting concerns) modüler biçimde uygulamak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** İçteki `wrapper` fonksiyonuna `@wraps(func)` uygulanmazsa sarmalanan fonksiyonun `__name__`, `__doc__` ve `__annotations__` gibi üstverileri kaybolur ve `wrapper` olarak görünür.


In [ ]:
from collections.abc import Callable
from functools import wraps
from time import perf_counter, sleep


def measure_time[**P, R](func: Callable[P, R]) -> Callable[P, R]:
    @wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        start: float = perf_counter()
        result: R = func(*args, **kwargs)
        elapsed: float = round(perf_counter() - start, 4)
        print(f'{func.__name__} executed in {elapsed}s')
        return result

    return wrapper


@measure_time
def export_audio(filename: str) -> str:
    sleep(0.1)
    return f'Exported {filename}'


print(export_audio('master.flac'))
print(export_audio.__name__)


## 10. Decorators with Arguments

Dekoratörün kendisine yapılandırma parametreleri (`@repeat(times=3)` gibi) geçirebilmek için standart dekoratör yapısının dışına bir fabrika fonksiyonu (decorator factory) katmanı eklenerek oluşturulan üç katmanlı yapılardır.

**Syntax**
```python
def decorator_factory(param):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            return func(*args, **kwargs)
        return wrapper
    return decorator
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Tekrar sayısı, zaman aşımı süresi, önbellek boyutu veya yetki rolü gibi özelleştirilebilir parametreler alan dekoratörler yazarken kullanılır.
* **Dikkat Edilmesi Gerekenler:** En dıştaki fonksiyon argümanları alır ve asıl dekoratörü döndürür; orta katman `func` nesnesini alır ve `wrapper`'ı döndürür; en içteki `wrapper` ise çağrı anındaki `*args, **kwargs` değerlerini işler.


In [ ]:
from collections.abc import Callable
from functools import wraps  # noqa: F811


def repeat(times: int, label: str = 'RUN') -> Callable[[Callable[..., object]], Callable[..., object]]:
    def decorator(func: Callable[..., object]) -> Callable[..., object]:
        @wraps(func)
        def wrapper(*args: object, **kwargs: object) -> object:
            last_result: object = None
            for step in range(1, times + 1):
                print(f'[{label} {step}/{times}]', end=' ')
                last_result = func(*args, **kwargs)
            return last_result

        return wrapper

    return decorator


@repeat(times=3, label='PING')
def ping_host(host: str) -> str:
    print(f'Checking {host}')
    return 'ALIVE'


ping_host('10.0.0.1')


## 11. Memoization (`@functools.cache`)

Maliyetli veya özyinelemeli (recursive) fonksiyon çağrılarının sonuçlarını girdi argümanlarına göre bellekte saklayarak (caching), aynı argümanlarla yapılan sonraki çağrılarda yeniden hesaplama yapmadan önbellekten döndüren optimizasyon tekniğidir.

**Syntax**
```python
from functools import cache, lru_cache

@cache
def compute(n: int) -> int:
    ...
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Üstel zaman karmaşıklığına ($O(2^n)$) sahip özyinelemeli algoritmaları (Fibonacci, dinamik programlama) doğrusal zamana ($O(n)$) düşürmek için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Zaman-bellek takası (time-memory tradeoff) söz konusudur; sınırsız önbellek (`@cache`) uzun süren süreçlerde RAM tüketimini artırabilir, bu durumda `@lru_cache(maxsize=128)` tercih edilmelidir. Ayrıca fonksiyon argümanlarının hash'lenebilir (immutable) olması zorunludur ve yalnızca saf (pure) fonksiyonlarda kullanılmalıdır.


In [ ]:
from functools import cache
from time import perf_counter  # noqa: F811


@cache
def fibonacci(n: int) -> int:
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)


start: float = perf_counter()
result: int = fibonacci(100)
elapsed: float = perf_counter() - start

print(f'Fib(100) = {result}')
print(f'Elapsed: {elapsed:.6f}s')
print(fibonacci.cache_info())


## 12. Context Managers (`with`, `__enter__`, `__exit__`)

`with` bloğuna girildiğinde kaynak tahsisini (`__enter__`), bloktan normal şekilde çıkıldığında veya bir istisna oluştuğunda kaynak temizliğini (`__exit__`) garanti eden protokol yapısıdır.

**Syntax**
```python
class Resource:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb) -> bool:
        return False
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Dosya akışları, veritabanı bağlantıları, ağ soketleri ve iş parçacığı kilitleri (locks) gibi kapatılması zorunlu kaynakları sızıntısız yönetmek için kullanılır.
* **Dikkat Edilmesi Gerekenler:** `__exit__` metodu `True` döndürürse blok içinde oluşan istisnayı bastırır (yutar); `False` veya `None` döndürürse temizlik işlemini yaptıktan sonra istisnay yukarı iletir. Basit durumlarda sınıf yazmak yerine `@contextlib.contextmanager` dekoratörü ve `try...finally` bloğu kullanılabilir.


In [ ]:
from types import TracebackType


class SessionConnection:
    def __init__(self, endpoint: str) -> None:
        self.endpoint = endpoint

    def __enter__(self) -> 'SessionConnection':
        print(f'Opening connection to {self.endpoint}')
        return self

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc_val: BaseException | None,
        exc_tb: TracebackType | None,
    ) -> bool:
        print(f'Closing connection to {self.endpoint}')
        return False


with SessionConnection('postgres://localhost:5432/app') as conn:
    print(f'Querying {conn.endpoint}')

**3.11+ Python:**

__enter__ gibi mevcut örneği (self) döndüren metotların dönüş tipi olarak sınıf adını string halinde ('SessionConnection') yazmak yerine Python 3.11 ile standart kütüphaneye eklenen typing.Self tipini kullanman gerekir. Sınıf adını doğrudan yazarsan ve ileride SessionConnection sınıfından kalıtım alan bir alt sınıf türetirsen, statik tip denetleyicileri alt sınıfın __enter__ metodunun alt sınıfı değil üst sınıfı döndürdüğünü varsayar. Self ise çağrıldığı sınıfın veya alt sınıfın tipini dinamik olarak korur.

```python
from types import TracebackType
from typing import Self

class SessionConnection:
    def __init__(self, endpoint: str) -> None:
        self.endpoint = endpoint

    def __enter__(self) -> Self:
        print(f'Opening connection to {self.endpoint}')
        return self

    def __exit__(
        self,
        exc_type: type[BaseException] | None,
        exc_val: BaseException | None,
        exc_tb: TracebackType | None,
    ) -> bool:
        print(f'Closing connection to {self.endpoint}')
        return False

```


## 13. Timing Code Performance (`perf_counter` & `timeit`)

Kod parçacıklarının veya fonksiyonların yürütme süresini ölçerek farklı algoritmaların hızını nesnel biçimde karşılaştırma yöntemleridir.

**Syntax**
```python
from time import perf_counter
import timeit

t0 = perf_counter()
...
elapsed = perf_counter() - t0

timings = timeit.repeat(stmt='...', repeat=5, number=100_000, globals=globals())
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Tek seferlik blok/G/Ç sürelerini ölçmek için `time.perf_counter()`, mikro kıyaslamalarda (micro-benchmarking) fonksiyonu yüz binlerce kez çalıştırıp en iyi süreyi bulmak için `timeit.repeat()` kullanılır.
* **Dikkat Edilmesi Gerekenler:** Duvar saati zamanı (wall-clock) yerine yüksek çözünürlüklü monotonik sayaç olan `perf_counter()` tercih edilmelidir. `timeit.repeat()` çıktısında arka plan işletim sistemi gürültüsünü elemek için ortalama yerine `min()` değeri baz alınmalıdır.


In [ ]:
import timeit


def build_with_loop(n: int) -> list[int]:
    items: list[int] = []
    for i in range(n):
        items.append(i * 2)
    return items


def build_with_comp(n: int) -> list[int]:
    return [i * 2 for i in range(n)]


def benchmark(stmt: str, repeat: int = 5, number: int = 50_000) -> float:
    best: float = min(timeit.repeat(stmt=stmt, repeat=repeat, number=number, globals=globals()))
    print(f'{stmt:<25} -> {best:.4f}s ({repeat * number:,} runs)')
    return best


benchmark('build_with_loop(100)')
benchmark('build_with_comp(100)')


## 14. Optimizing Code (Best Practices)

Kod optimizasyonu, çalışan ve doğrulanmış bir yazılımın gerçek darboğazlarını (bottlenecks) ölçümleyerek yalnızca anlamlı kazanç sağlayacak noktalarda iyileştirme yapma disiplinidir.

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Erken Optimizasyondan Kaçınma:** Önce okunabilir, hatasız ve tüm gereksinimleri karşılayan çalışan kod yazılmalı; optimizasyona yalnızca profilleme (`cProfile`, `timeit`) sonuçları gerçek bir performans sorunu gösterdiğinde geçilmelidir.
* **Algoritmik ve Yerleşik Kazançlar:** Nanosaniyelik mikro hileler yerine doğru veri yapısını seçmek (üyelik sorgusunda `list` yerine `set`/`dict` kullanmak, yerleşik C fonksiyonlarını ve üreteçleri tercih etmek) çok daha yüksek performans artışı sağlar.


In [ ]:
from time import perf_counter

data_list: list[int] = list(range(200_000))
data_set: set[int] = set(data_list)
target: int = 199_999

t0: float = perf_counter()
_ = target in data_list
list_time: float = perf_counter() - t0

t1: float = perf_counter()
_ = target in data_set
set_time: float = perf_counter() - t1

print(f'List lookup: {list_time:.6f}s | Set lookup: {set_time:.6f}s')


## 15. Monkey Patching

Bir modülün, sınıfın veya nesnenin özniteliklerini ya da metotlarını kaynak kodunu değiştirmeden çalışma zamanında (runtime) dinamik olarak değiştirme veya üzerine yazma tekniğidir.

**Syntax**
```python
target_module.target_function = custom_replacement_function
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** Birim testlerinde ücretli veya yavaş dış API çağrılarını sahte (mock) yanıtlarla değiştirmek ya da üçüncü parti bir kütüphanedeki kritik bir hatayı geçici olarak yamamak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Üretim kodunda yaygın kullanımı kod takibini zorlaştırır ve kütüphane güncellemelerinde sessiz kırılmalara yol açar. Test ortamlarında kalıcı yan etkileri önlemek için `unittest.mock.patch` veya `pytest` `monkeypatch` fikstürü tercih edilmelidir.


In [ ]:
class WeatherClient:
    def fetch(self, city: str) -> dict[str, object]:
        raise ConnectionError('External API unreachable in offline test')


def mock_fetch(self: WeatherClient, city: str) -> dict[str, object]:
    return {'city': city, 'temp_c': 21, 'source': 'mock'}


client: WeatherClient = WeatherClient()
WeatherClient.fetch = mock_fetch

print(client.fetch('Istanbul'))


## 16. Custom Exceptions

Yerleşik `Exception` sınıfından kalıtım alarak uygulamaya özgü hata durumlarını, ek hata kodlarını ve özel metotları taşıyabilen kendi istisna sınıflarımızı tanımlama yöntemidir.

**Syntax**
```python
class CustomError(Exception):
    def __init__(self, message: str, code: int) -> None:
        self.code = code
        super().__init__(message, code)
```

**Kullanım Alanları ve Dikkat Edilmesi Gerekenler**
* **Ne Zaman Kullanılır:** İş mantığına özgü hataları (`InsufficientBalanceError`, `NegativeValueError`) standart Python hatalarından ayırıp `except` bloklarında hassas biçimde yakalamak için kullanılır.
* **Dikkat Edilmesi Gerekenler:** Özel istisna sınıfları `BaseException` yerine doğrudan `Exception` sınıfından türetilmelidir (`KeyboardInterrupt` veya `SystemExit` gibi sistem sinyallerini yanlışlıkla yakalamamak için). Özel `__init__` imzasına sahip istisnaların `pickle` ile serileştirilebilmesi gerekiyorsa `__reduce__` dunder metodu tanımlanmalıdır.


In [ ]:
class NegativeValueError(Exception):
    def __init__(self, value: float, error_code: int) -> None:
        self.value = value
        self.error_code = error_code
        super().__init__(f'{value} is below zero (Code: {error_code})', value, error_code)

    def __str__(self) -> str:
        return f'NegativeValueError [{self.error_code}]: {self.value} is not allowed'

    def __reduce__(self) -> tuple[type['NegativeValueError'], tuple[float, int]]:
        return (type(self), (self.value, self.error_code))


def set_gain(db_value: float) -> None:
    if db_value < 0:
        raise NegativeValueError(db_value, 1004)


try:
    set_gain(-6.5)
except NegativeValueError as err:
    print(err)
    print(err.args)
